# LoRA Fine-Tuning Assignment

## Objective

Apply what you learned in the SFT + LoRA tutorial to fine-tune **TinyLlama-1.1B** on the **databricks-dolly-15k** dataset using LoRA.

## What is provided

- All installation, import, and utility cells are provided.
- Each task cell has a comment block describing exactly what you need to write.

## What you need to do

Fill in every cell marked with `# YOUR CODE HERE`. There are **8 tasks** in total.

## Prerequisites

- You have completed the SFT + LoRA tutorial notebook.
- Runtime: **GPU** (T4 is sufficient).

---
## Part 0 — Setup (provided)

In [ ]:
!pip install -U transformers trl peft datasets accelerate torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
pip install torch==2.11.0 torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 39.2 MB/s eta 0:00:00
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.10.0+cu128
    Uninstalling torchaudio-2.10.0+cu128:
      Successfully uninstalled torchaudio-2.10.0+cu128
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.25.0+cu128
    Uninstalling torchvision-0.25.0+cu128:
      Successfully uninstalled torchvision-0.25.0+cu128


In [ ]:
#!apt-get install

In [ ]:
import torch
import torchvision

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA version: {torch.version.cuda}")

PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cpu
CUDA available: True
PyTorch CUDA version: 13.0


In [ ]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


### Utility functions (provided)

In [ ]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

---
## Part 1 — Configuration

### Task 1: Define all hyperparameters

Set the following configuration variables:

| Variable | Value |
|---|---|
| `MODEL_NAME` | `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"` |
| `DATASET_NAME` | `"databricks/databricks-dolly-15k"` |
| `NUM_TRAIN_SAMPLES` | `1000` |
| `MAX_SEQ_LENGTH` | `512` |
| `LORA_R` | `8` |
| `LORA_ALPHA` | `16` |
| `LORA_DROPOUT` | `0.05` |
| `LORA_TARGET_MODULES` | `["q_proj", "v_proj"]` (attention only) |
| `LEARNING_RATE` | `2e-4` |
| `NUM_EPOCHS` | `1` |
| `BATCH_SIZE` | `4` |
| `GRAD_ACCUM_STEPS` | `4` |
| `OUTPUT_DIR` | `"./sft-lora-tinyllama"` |

In [ ]:
# YOUR CODE HERE — define all configuration variables listed above
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # Base model (no -Instruct suffix!)
DATASET_NAME = "databricks/databricks-dolly-15k" # Classic instruction-tuning dataset
NUM_TRAIN_SAMPLES = 1000 # Subset for fast training on Colab
MAX_SEQ_LENGTH = 512 # Maximum sequence length

# LoRA hyperparameters
LORA_R = 8 # Rank
LORA_ALPHA = 16 # Scaling factor (2x rank)
LORA_DROPOUT = 0.05 # Regularization
LORA_TARGET_MODULES = ["q_proj", "v_proj"] #attention only

# Training hyperparameters
LEARNING_RATE = 2e-4 # 10x higher than full fine-tuning
NUM_EPOCHS = 1 # one complete pass of the entire training dataset through the model
BATCH_SIZE = 4 # number of training samples processed in one forward/backward pass before updating the model's internal parameters (weights)
GRAD_ACCUM_STEPS = 4 #  Effective batch size = 4 * 4 = 16
OUTPUT_DIR = "./sft-lora-tinyllama"


---
## Part 2 — Load Model and Tokenizer

### Task 2: Load the tokenizer and base model

1. Load the tokenizer from `MODEL_NAME`.
2. If the tokenizer has no pad token, set it to the eos token.
3. Load the model in `float16` with `device_map="auto"`.
4. Print the total parameter count.

In [ ]:
from google.colab import userdata
import os

# This retrieves the token from your Colab Secrets
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
# YOUR CODE HERE — load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Model Max Length: {tokenizer.model_max_length:,}")
print(f"Pad Token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

# Model in float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

total_params = sum(p.numel() for p in model.parameters())
model_size_gb = sum(p.numel() for p in model.parameters()) / 1024**3

print(f"\nBase Model Loaded : {MODEL_NAME}")
print(f"Total Parameters: {total_params:,}")
print(f"Model Size (GB): {model_size_gb:.2f} GB")
print(f"GPU Memory Allocated : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Vocabulary size: 32,000
Model Max Length: 2,048
Pad Token: '</s>' (id=2)


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Base Model Loaded : TinyLlama/TinyLlama-1.1B-Chat-v1.0
Total Parameters: 1,100,048,384
Model Size (GB): 1.02 GB
GPU Memory Allocated : 2.05 GB


---
## Part 3 — Test Base Model

### Task 3: Generate responses from the base model

Use the provided `generate_response` function with these test prompts:
1. `"Explain what a neural network is in 2-3 sentences."`
2. `"What are three tips for better sleep?"`

Store the responses in a list called `base_responses` for later comparison.

In [ ]:
# YOUR CODE HERE — define test_prompts, generate base model responses, store in base_responses
test_prompts = [
    "Explain what a neural network is in 2-3 sentences.",
    "What are three tips for better sleep?"
]

print("="*70)
print("BASE MODEL RESPONSES (Before SFT)")
print("="*70)
print("\n")
for i, prompt in enumerate(test_prompts):
    print(f"?" *50)
    print(f"Prompt {i+1}: {prompt}")
    print(f"?" *50)
    response = generate_response(model, tokenizer, prompt, max_new_tokens=200)
    print(f"Response: {response[:500]}\n")
    base_responses = []
    for prompt in test_prompts:
        base_responses.append(response)

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL RESPONSES (Before SFT)


??????????????????????????????????????????????????
Prompt 1: Explain what a neural network is in 2-3 sentences.
??????????????????????????????????????????????????


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: A neural network is a type of artificial intelligence (AI) system that uses a combination of artificial and human-like logic to make predictions and make decisions. It consists of a series of interconnected units, called nodes, which are connected through weighted connections. These connections allow nodes to learn from each other and improve their ability to make predictions and make decisions based on the data they are given. Neural networks are used in a wide range of applications, including 

??????????????????????????????????????????????????
Prompt 2: What are three tips for better sleep?
??????????????????????????????????????????????????
Response: 1. Establish a consistent sleep routine: Try to go to bed and wake up at the same time every day, even on weekends.

2. Create a comfortable sleep environment: Make sure your bedroom is cool, dark, and quiet. Keep the bedroom temperature at a comfortable level for you.

3. Limit caffeine and alcohol: Both of these substances c

---
## Part 4 — Prepare the Dataset

### Task 4: Load and format the Dolly dataset

The Dolly dataset has columns: `instruction`, `context`, `response`, `category`.

1. Load the dataset (`split="train"`).
2. Write a function `format_dolly_to_chat(example)` that converts each example to:
   ```python
   {"messages": [
       {"role": "user", "content": <instruction + context if present>},
       {"role": "assistant", "content": <response>}
   ]}
   ```
   If `context` is non-empty, concatenate it to the instruction with a newline separator.
3. Shuffle with `seed=42`, select `NUM_TRAIN_SAMPLES` examples, and apply the formatting function.
4. Print the size of the formatted dataset and one example.



```
Example formatted entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}
```



In [ ]:
# YOUR CODE HERE — load dolly dataset, write format_dolly_to_chat, apply it
# Load dataset
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Full dataset size: {len(raw_dataset):,} examples")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nExample entry:")
print(raw_dataset[0])

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Full dataset size: 15,011 examples
Columns: ['instruction', 'context', 'response', 'category']

Example entry:
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [ ]:
# write format_dolly_to_chat, apply it
def format_dolly_to_chat(example):
    """
    Convert Dolly format to chat format.('instruction', 'context', 'response', 'category') to conversational messages format for SFTTrainer.

    Dolly format:
    {"instruction": "...", "context": "...", "response": "...", "category": "..."}

    Chat format:
    {"messages": [
    {"role": "user", "content": <instruction + context if present>},
    {"role": "assistant", "content": <response>}
]}
    """
    # Combine Instruction and Context (if present)
    if example.get("context") and example["context"].strip():
        instruction = f"{example['instruction']}\n\n{example['context']}"
    else:
        instruction = example['instruction']

    return {"messages": [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": example['response']},
    ]}

# Applying formatting and taking a subset for training
dataset = raw_dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))
dataset = dataset.map(
    format_dolly_to_chat,
    remove_columns = raw_dataset.column_names,
    desc = "Formatting to chat messages",
    )
print(f"Formatted dataset size: {len(dataset):,} examples")
print(f"Columns: {dataset.column_names}")
print(f"\nExample entry:")
print(dataset[0])

Formatting to chat messages:   0%|          | 0/1000 [00:00<?, ? examples/s]

Formatted dataset size: 1,000 examples
Columns: ['messages']

Example entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}


In [ ]:
# Verify the chat template renders correctly
sample = dataset[0]
rendered = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
print("Rendered chat template (first example):\n")
print(rendered[:800])

Rendered chat template (first example):

<|user|>
Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?</s>
<|assistant|>
Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree</s>



---
## Part 5 — Configure LoRA

### Task 5: Create the LoRA config and inspect trainable parameters

1. Create a `LoraConfig` using the hyperparameters from Task 1. Set `bias="none"` and `task_type="CAUSAL_LM"`.
2. Apply it to the model with `get_peft_model`.
3. Use `print_trainable_params` to display the trainable vs total parameters.
4. Print the LoRA-wrapped structure of one attention layer (e.g., `q_proj` of layer 0).

In [ ]:
# YOUR CODE HERE — create LoraConfig, apply to model, print trainable params

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

print("===== LORA CONFIGURATION =====")
print(f" Rank (r) : {peft_config.r}")
print(f" Alpha (?) : {peft_config.lora_alpha}")
print(f" Effective Scaling : {peft_config.lora_alpha / peft_config.r}")
print(f" Dropout : {peft_config.lora_dropout}")
print(f" Target Modules : {peft_config.target_modules}")
print(f" Bias : {peft_config.bias}")
print(f" Task Type : {peft_config.task_type}")
print("==============================\n")

lora_model = get_peft_model(model, peft_config)
print("\n" + "=" * 50)
print("PARAMETER COMPARISON")
print("=" * 50)
print_trainable_params(lora_model)

print("\n\n======= LoRA-WRAPPED STRUCTURE =======")
print("LoRA Module Example (q_proj of layer 0)")
print("======================================\n")
print(lora_model.model.model.model.layers[0].self_attn.q_proj)
print("==============================\n")

===== LORA CONFIGURATION =====
 Rank (r) : 8
 Alpha (?) : 16
 Effective Scaling : 2.0
 Dropout : 0.05
 Target Modules : {'q_proj', 'v_proj'}
 Bias : none
 Task Type : CAUSAL_LM


PARAMETER COMPARISON
Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784


======= LoRA-WRAPPED STRUCTURE =======
LoRA Module Example (q_proj of layer 0)

lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)



---
## Part 6 — Train

### Task 6: Set up SFTTrainer and run training

1. Delete the PEFT-wrapped model and call `clear_memory()`. Reload the base model fresh (SFTTrainer applies LoRA itself via `peft_config`).
2. Create an `SFTConfig` with the hyperparameters from Task 1. Include:
   - Cosine LR scheduler, warmup ratio 0.03, weight decay 0.01
   - `gradient_checkpointing=True`
   - bf16 if supported, else fp16
   - `max_length=MAX_SEQ_LENGTH`
   - Logging every 10 steps
3. Create an `SFTTrainer` with the model, training args, dataset, tokenizer, and `peft_config`.
4. Call `trainer.train()` and print the final loss.

In [ ]:
# Clean Up
del lora_model
clear_memory()

# Reload Model fresh for SFTTrainer
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

GPU allocated: 2.06 GB
GPU reserved:  2.10 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# SFTTraining Configuration
training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_length=MAX_SEQ_LENGTH,
    logging_steps=10,
    gradient_checkpointing=True,
    seed=42,
)

print("===== SFT TRAINING CONFIGURATION =====")
print(f" Batch Size : {training_config.per_device_train_batch_size}")
print(f" Epochs : {training_config.num_train_epochs}")
print(f" Learning Rate : {training_config.learning_rate}")
print(f" Weight Decay : {training_config.weight_decay}")
print(f" Gradient Accumulation : {training_config.gradient_accumulation_steps}")
print(f" LR Scheduler : {training_config.lr_scheduler_type}")
print(f" Warmup Ratio : {training_config.warmup_ratio}")
print(f" FP16 : {training_config.fp16}")
print(f" BF16 : {training_config.bf16}")
print(f" Max Seq Length : {training_config.max_length}")
print(f" Logging Steps : {training_config.logging_steps}")
print(f" Gradient Checkpointing : {training_config.gradient_checkpointing}")
print(f" Max Steps : {training_config.max_steps}")
print(f" Push to Hub : {training_config.push_to_hub}")
print(f" Report to : {training_config.report_to}")
print(f" Optimizer : {training_config.optim}")
print(f" Logging Dir : {training_config.logging_dir}")
print(f" Save Strategy : {training_config.save_strategy}")
print(f" Save Steps : {training_config.save_steps}")
print(f" Save Total Limit : {training_config.save_total_limit}")
print(f" Seed : {training_config.seed}")
print(f" Data Seed : {training_config.data_seed}")
print("======================================\n")

===== SFT TRAINING CONFIGURATION =====
 Batch Size : 4
 Epochs : 1
 Learning Rate : 0.0002
 Weight Decay : 0.01
 Gradient Accumulation : 4
 LR Scheduler : SchedulerType.COSINE
 Warmup Ratio : None
 FP16 : False
 BF16 : True
 Max Seq Length : 512
 Logging Steps : 10
 Gradient Checkpointing : True
 Max Steps : -1
 Push to Hub : False
 Report to : []
 Optimizer : OptimizerNames.ADAMW_TORCH_FUSED
 Logging Dir : None
 Save Strategy : SaveStrategy.STEPS
 Save Steps : 500
 Save Total Limit : None
 Seed : 42
 Data Seed : None



In [ ]:
#SFTTrainer, train
trainer = SFTTrainer(
    model=model,
    args = training_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
10,2.098832
20,1.862427
30,1.749236
40,1.726048
50,1.770034
60,1.714174


TrainOutput(global_step=63, training_loss=1.8193169321332658, metrics={'train_runtime': 1075.1626, 'train_samples_per_second': 0.93, 'train_steps_per_second': 0.059, 'total_flos': 2189307248934912.0, 'train_loss': 1.8193169321332658})

---
## Part 7 — Save the Adapter

### Task 7: Save the LoRA adapter and print its size

1. Save the trained model to `f"{OUTPUT_DIR}/final-adapter"`.
2. Also save the tokenizer to the same path.
3. Compute and print the adapter size in MB.

In [ ]:
# YOUR CODE HERE — save adapter and tokenizer, print adapter size
adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

adapter_size = sum(os.path.getsize(os.path.join(adapter_path, f))
for f in os.listdir(adapter_path)
if os.path.isfile(os.path.join(adapter_path, f))
)

print("Adapter Saved to : {adapter_path}")
print(f"Adapter Size: {adapter_size / 1024**2:.2f} MB")
print(f"(Compare to Full Model: ~{sum(p.numel()*2 for p in model.parameters()) / 1024**3:.1f} GB)")

Adapter Saved to : {adapter_path}
Adapter Size: 7.77 MB
(Compare to Full Model: ~2.1 GB)


---
## Part 8 — Evaluate

### Task 8: Compare base vs fine-tuned responses

1. Use `trainer.model` (already has LoRA weights) to generate responses for the same `test_prompts`.
2. Print a side-by-side comparison of base vs fine-tuned responses.

In [ ]:
# YOUR CODE HERE — generate fine-tuned responses and print comparison
FineTuned_Model = trainer.model
FineTuned_Model.eval()

print("="*70)
print("FINE-TUNED MODEL RESPONSES (After SFT with LoRA)")
print("="*70)
print("\n")

FineTuned_Responses = []
for i, prompt in enumerate(test_prompts):
  print(f"?" *50)
  print(f"Prompt {i+1}: {prompt}")
  print(f"?" *50)
  response = generate_response(FineTuned_Model, tokenizer, prompt, max_new_tokens=200)
  print(f"Response: {response[:500]}\n")
  FineTuned_Responses.append(response)
  print("="*70)


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL RESPONSES (After SFT with LoRA)


??????????????????????????????????????????????????
Prompt 1: Explain what a neural network is in 2-3 sentences.
??????????????????????????????????????????????????


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: A neural network is a mathematical model that simulates the behavior of a biological neuron. It consists of a set of interconnected nodes (neurons) that can take on different states (inputs) and can be connected to other neurons (outputs). The neurons are fed with the input signals and the output is produced based on the weighted connections between the neurons. A neural network can perform a variety of tasks, including image recognition, speech recognition, and learning.

??????????????????????????????????????????????????
Prompt 2: What are three tips for better sleep?
??????????????????????????????????????????????????
Response: 1. Create a consistent sleep schedule.
2. Create a relaxing bedtime routine.
3. Avoid electronics and caffeine before bedtime.

I hope this helps!



In [ ]:
print("="*70)
print("COMPARISON: BASE MODEL RESPONSES (Before SFT) vs. FINE-TUNED MODEL (LoRA SFT)")
print("="*70)
print("\n")
for i, prompt in enumerate(test_prompts):
  print(f"?" *50)
  print(f"Prompt {i+1}: {prompt}")
  print(f"?" *50)
  print(f"Base Model Response:\n\n {base_responses[i:500]}")
  print(f"\nFine-Tuned Model Response:\n\n {FineTuned_Responses[i:500]}\n")

COMPARISON: BASE MODEL RESPONSES (Before SFT) vs. FINE-TUNED MODEL (LoRA SFT)


??????????????????????????????????????????????????
Prompt 1: Explain what a neural network is in 2-3 sentences.
??????????????????????????????????????????????????
Base Model Response:

 ['1. Establish a consistent sleep routine: Try to go to bed and wake up at the same time every day, even on weekends.\n\n2. Create a comfortable sleep environment: Make sure your bedroom is cool, dark, and quiet. Keep the bedroom temperature at a comfortable level for you.\n\n3. Limit caffeine and alcohol: Both of these substances can disrupt your sleep patterns. Try to avoid them before bed.\n\n4. Avoid electronics before bed: Electronic devices like smartphones, tablets, and computers can interfere with sleep. Turn them off at least an hour before bed.\n\n5. Exercise regularly: Exercise before bed can help you relax and fall asleep faster. However, do not exercise too close to bedtime, as it can interfere with your sleep.'

---
## Done

You have successfully fine-tuned TinyLlama with LoRA on the Dolly dataset. Review your base vs fine-tuned outputs and note the differences in instruction-following quality.